In [1]:
!pip install transformers
!pip install scikit-learn
!pip install pandas tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModel
import torch

In [4]:
df = pd.read_csv("/content/drive/MyDrive/DATASET_TESIS_CIC/CLASIFICACIÓN/CB_DATASET_MULTICLASS-MULTILABEL.csv", encoding="utf-8")

In [5]:
# Validación de columnas
df = df[df['category_single'].notnull()].reset_index(drop=True)
texts = df['text_cleaned'].astype(str).tolist()
labels = df['category_single'].tolist()

In [6]:
# Cargar embeddings de BETO
tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased")
model = AutoModel.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #optimizar el espacio
model.to(device)
model.eval()

# Extraer embeddings con CLS
def get_cls_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

# Convertir todos los textos
embeddings = []
for text in tqdm(texts, desc="Extrayendo embeddings"):
    emb = get_cls_embedding(text)
    embeddings.append(emb)

X = np.array(embeddings)
y = np.array(labels)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/486k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Extrayendo embeddings:   0%|          | 1/166869 [00:00<13:08:59,  3.52it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Extrayendo embeddings:   5%|▌         | 8835/166869 [21:39<6:27:25,  6.80it/s]


KeyboardInterrupt: 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
svm_model = LinearSVC(class_weight="balanced", random_state=42)
svm_model.fit(X_train, y_train)

In [ ]:
y_pred = svm_model.predict(X_test)
print(classification_report(y_test, y_pred))